# C1. From commutator equations to PDEs, and the bracket of the κ-identification

Companion notebook to the bachelor's thesis *El formalismo ODM en sistemas híbridos clásico-cuánticos* (Santiago Puyol Miano, Universidad de Zaragoza, 2026).

ODM starts from Ehrenfest-type commutator equations and turns them into partial differential equations for the generator of the dynamics. The tool is the chain rule for commutators: if every commutator $[\hat A_k,\hat B]$ is a scalar, then $[f(\hat A_1,\ldots,\hat A_n),\hat B]=\sum_k[\hat A_k,\hat B]\,\partial_kf(\hat A_1,\ldots,\hat A_n)$.

This notebook checks

1. the Poisson bracket $\{x_q,p_q\}_\Xi=\hbar\kappa$ of the κ-identification;
2. the special cases of the chain rule used in the derivations;
3. the quantum Hamiltonian $\hat H=\hat p^2/2m+U(\hat x)$, and the obstruction that rules out a classical generator built from $\hat x,\hat p$ alone;
4. the Liouvillian $\hat L=(\hat p/m)\hat\lambda_x-U'(\hat x)\hat\lambda_p$ and the Liouville equation it gives.

The chain-rule checks use polynomial test functions, and the derivations of $\hat H$ and $\hat L$ use the harmonic oscillator $U=\tfrac12m\omega^2x^2$.

The κ-identification is a linear map that is not symplectic, and its bracket $\hbar\kappa$ is the source of the commutator $[\hat x_q,\hat p_q]=i\hbar\kappa$. The symplectic transformation $g_\alpha$ of C2, which satisfies $g_\alpha^\top Jg_\alpha=J$, is a different map.

**Kernel:** SageMath 10.8.

## Notation

- Phase space $\mathcal P=\mathbb R^2$ with $[\hat q,\hat p]=i\hbar$. In the code its position is `q`.
- $\Xi=T^*\mathcal P\cong\mathbb R^4$ with coordinates $(x,p,\lambda_x,\lambda_p)$, symplectic form $\Omega=dx\wedge d\lambda_x+dp\wedge d\lambda_p$ and potential $\Theta=\lambda_x\,dx+\lambda_p\,dp$.
- On $\Xi$, $\hat\lambda_x=-i\partial_x$ and $\hat\lambda_p=-i\partial_p$, so $[\hat x,\hat\lambda_x]=[\hat p,\hat\lambda_p]=i$ with unit $1$, not $\hbar$.
- κ-identification, with $\kappa\in[0,1]$ the control parameter ($\kappa=0$ classical, $\kappa=1$ quantum):
  $\hat x_q=\hat x-\tfrac{\hbar\kappa}{2}\hat\lambda_p$, $\hat p_q=\hat p+\tfrac{\hbar\kappa}{2}\hat\lambda_x$.

In [1]:
var('x p lambda_x lambda_p hbar kappa m omega')
assume(hbar > 0, kappa >= 0, kappa <= 1, m > 0, omega > 0)

# kappa-identification
x_q  = x - hbar*kappa/2 * lambda_p
p_q  = p + hbar*kappa/2 * lambda_x
th_x = lambda_x          # vartheta_x = lambda_x
th_p = lambda_p          # vartheta_p = lambda_p

# Poisson bracket on Xi = R^4, coordinates (x, p, lambda_x, lambda_p).
# Canonical: {x,lambda_x} = {p,lambda_p} = 1, all other brackets 0.
def poisson_Xi(f, g):
    """Poisson bracket {f,g}_Xi = Omega(X_f, X_g), Omega = dx^dlambda_x + dp^dlambda_p."""
    coords  = [x, p]
    momenta = [lambda_x, lambda_p]
    result = SR(0)
    for qi, pi in zip(coords, momenta):
        result += diff(f, qi)*diff(g, pi) - diff(f, pi)*diff(g, qi)
    return result

print("x_q =", x_q, "   p_q =", p_q)

x_q = -1/2*hbar*kappa*lambda_p + x    p_q = 1/2*hbar*kappa*lambda_x + p


## 1. The bracket of the κ-identification

The Poisson bracket on $\Xi$ is

$$\{f,g\}_\Xi = \frac{\partial f}{\partial x}\frac{\partial g}{\partial\lambda_x}
  - \frac{\partial f}{\partial\lambda_x}\frac{\partial g}{\partial x}
  + \frac{\partial f}{\partial p}\frac{\partial g}{\partial\lambda_p}
  - \frac{\partial f}{\partial\lambda_p}\frac{\partial g}{\partial p},$$

with canonical brackets $\{x,\lambda_x\}=\{p,\lambda_p\}=1$ and all others zero. For the κ-identification

$$\{x_q,p_q\}_\Xi=\hbar\kappa,$$

so the identification is not symplectic, and this bracket gives the commutator $[\hat x_q,\hat p_q]=i\hbar\kappa$.

In [2]:
# Canonical brackets
assert poisson_Xi(x, lambda_x) == 1,  f"{{x,lambda_x}} = {poisson_Xi(x, lambda_x)}"
assert poisson_Xi(p, lambda_p) == 1,  f"{{p,lambda_p}} = {poisson_Xi(p, lambda_p)}"
assert poisson_Xi(x, p)        == 0,  f"{{x,p}} = {poisson_Xi(x, p)}"
assert poisson_Xi(x, lambda_p) == 0,  f"{{x,lambda_p}} = {poisson_Xi(x, lambda_p)}"
assert poisson_Xi(p, lambda_x) == 0,  f"{{p,lambda_x}} = {poisson_Xi(p, lambda_x)}"
print("OK: {x,lambda_x} = {p,lambda_p} = 1 and the other brackets vanish")

bracket_xq_pq = poisson_Xi(x_q, p_q).simplify_full()

# Term by term:
#   dx_q/dx = 1,               dp_q/dlambda_x = hbar*kappa/2   ->  +hbar*kappa/2
#   dx_q/dlambda_p = -hbar*kappa/2,  dp_q/dp = 1               ->  +hbar*kappa/2
assert (bracket_xq_pq - hbar*kappa).is_zero(), \
    f"{{x_q,p_q}} = {bracket_xq_pq}, expected hbar*kappa"
print(f"OK: {{x_q, p_q}}_Xi = {bracket_xq_pq}")

OK: {x,lambda_x} = {p,lambda_p} = 1 and the other brackets vanish
OK: {x_q, p_q}_Xi = hbar*kappa


## 2. Operators on $\mathcal P$ and on $\Xi$

On $\mathcal P$: $\hat q=q$ and $\hat p=-i\hbar\,\partial_q$, acting on a test function $\psi_{\mathcal P}(q)$.
On $\Xi$: $\hat x=x$, $\hat p=p$, $\hat\lambda_x=-i\partial_x$ and $\hat\lambda_p=-i\partial_p$, acting on a test function $\psi_\Xi(x,p)$.
The two spaces use different position symbols (`q` and `x`) so that their operators are never mixed.

In [3]:
var('q')  # position on P (x is the position on Xi)

psi_P  = function('psi_P')(q)       # test function on P
psi_Xi = function('psi_Xi')(x, p)   # test function on Xi, independent of the lambdas

# Operators on P (physical hbar)
q_hat_P  = lambda f: q * f
p_hat_P  = lambda f: -I * hbar * diff(f, q)   # p_hat = -i*hbar*d/dq

# Operators on Xi (unit 1)
x_hat_Xi  = lambda f: x * f
p_hat_Xi  = lambda f: p * f
lx_hat    = lambda f: -I * diff(f, x)          # lambda_x_hat = -i*d/dx
lp_hat    = lambda f: -I * diff(f, p)          # lambda_p_hat = -i*d/dp

def comm_op(A, B, f):
    """Commutator [A,B] applied to f: A(B(f))-B(A(f)), simplified."""
    return (A(B(f)) - B(A(f))).simplify_full()

## 3. The chain rule for commutators

Let $\hat C_k=[\hat A_k,\hat B]$. If $[\hat A_k,\hat C_l]=[\hat B,\hat C_k]=0$ for all $k,l$, which holds in particular when every $\hat C_k$ is a scalar, then

$$[f(\hat A_1,\ldots,\hat A_n),\hat B]=\sum_k [\hat A_k,\hat B]\,\partial_k f(\hat A_1,\ldots,\hat A_n).$$

The notebook does not prove the rule. The cell below checks the four special cases that the derivations use.

| Check | Identity | Space | Used for |
|---|---|---|---|
| A1 | $[f(\hat q),\hat p]=i\hbar f'(\hat q)$ | $\mathcal P$ | quantum Hamiltonian |
| A2 | $[\hat p,\hat q]=-i\hbar$ | $\mathcal P$ | canonical commutator |
| A3 | $[f(\hat x),\hat\lambda_x]=if'(\hat x)$ | $\Xi$ | Liouvillian |
| A4 | $[h(\hat p),\hat\lambda_p]=ih'(\hat p)$ | $\Xi$ | Liouvillian |

In [4]:
# A1: [f(q_hat), p_hat] = i*hbar*f'(q)
f_test_q = q^2
lhs_A1 = comm_op(lambda f: f_test_q * f, p_hat_P, psi_P)
rhs_A1 = (I * hbar * diff(f_test_q, q) * psi_P).simplify_full()
assert (lhs_A1 - rhs_A1).simplify_full().is_zero(), \
    f'A1 FAILED: lhs={lhs_A1}, rhs={rhs_A1}'
print('OK (A1): [q^2, p_hat] = i*hbar*2q')

# A2: [p_hat, q_hat] = -i*hbar
lhs_A2 = comm_op(p_hat_P, q_hat_P, psi_P)
rhs_A2 = (-I * hbar * psi_P).simplify_full()
assert (lhs_A2 - rhs_A2).simplify_full().is_zero(), \
    f'A2 FAILED: lhs={lhs_A2}, rhs={rhs_A2}'
print('OK (A2): [p_hat, q_hat] = -i*hbar')

# A3: [f(x_hat), lx_hat] = i*f'(x)  (unit 1)
f_test_x = x^2
lhs_A3 = comm_op(lambda f: f_test_x * f, lx_hat, psi_Xi)
rhs_A3 = (I * diff(f_test_x, x) * psi_Xi).simplify_full()
assert (lhs_A3 - rhs_A3).simplify_full().is_zero(), \
    f'A3 FAILED: lhs={lhs_A3}, rhs={rhs_A3}'
print('OK (A3): [x^2, lambda_x_hat] = i*2x')

# A4: [h(p_hat), lp_hat] = i*h'(p)  (unit 1)
h_test_p = p^2
lhs_A4 = comm_op(lambda f: h_test_p * f, lp_hat, psi_Xi)
rhs_A4 = (I * diff(h_test_p, p) * psi_Xi).simplify_full()
assert (lhs_A4 - rhs_A4).simplify_full().is_zero(), \
    f'A4 FAILED: lhs={lhs_A4}, rhs={rhs_A4}'
print('OK (A4): [p^2, lambda_p_hat] = i*2p')

OK (A1): [q^2, p_hat] = i*hbar*2q
OK (A2): [p_hat, q_hat] = -i*hbar
OK (A3): [x^2, lambda_x_hat] = i*2x
OK (A4): [p^2, lambda_p_hat] = i*2p


## 4. The quantum Hamiltonian and the classical obstruction

**Quantum case.** With $[\hat x,\hat p]=i\hbar$, the chain rule turns the Ehrenfest equations into $H'_p=p/m$ and $H'_x=U'(x)$, which integrate to $\hat H=\hat p^2/2m+U(\hat x)$. The cell checks $[\hat H,\hat q]$ and $[\hat H,\hat p]$ for the oscillator, and that the two equations integrate to $H$.

**Classical case.** If $\hat x$ and $\hat p$ commute, the chain rule gives $[L(\hat x,\hat p),\hat x]=0$ for every function $L$, while the classical Ehrenfest equation requires $im[\hat L,\hat x]=\hat p\neq0$. No generator built from $\hat x,\hat p$ alone works, and one adjoins $\hat\lambda_x,\hat\lambda_p$ with $[\hat x,\hat\lambda_x]=[\hat p,\hat\lambda_p]=i$.

In [5]:
# Quantum case on P, harmonic oscillator H = -hbar^2/(2m) d^2/dq^2 + m*omega^2/2 * q^2
H_osc_P = lambda f: (-hbar^2/(2*m))*diff(f, q, 2) + (m*omega^2/2*q^2)*f

# (a) [H, q_hat] = -i*hbar*(p_hat/m) = -hbar^2/m * d/dq
lhs_Qa = comm_op(H_osc_P, q_hat_P, psi_P)
rhs_Qa = (-hbar^2/m * diff(psi_P, q)).simplify_full()
assert (lhs_Qa - rhs_Qa).simplify_full().is_zero(), \
    f'[H, q_hat] FAILED: {lhs_Qa} != {rhs_Qa}'
print("OK: [H_osc, q_hat] = -hbar^2/m * d/dq  (= -i*hbar*p_hat/m)")

# (b) [H, p_hat] = i*hbar*U'(q) = i*hbar*m*omega^2*q
lhs_Qb = comm_op(H_osc_P, p_hat_P, psi_P)
rhs_Qb = (I*hbar*m*omega^2*q * psi_P).simplify_full()
assert (lhs_Qb - rhs_Qb).simplify_full().is_zero(), \
    f'[H, p_hat] FAILED: {lhs_Qb} != {rhs_Qb}'
print("OK: [H_osc, p_hat] = i*hbar*U'(q)")

# (c) H'_p = p/m and H'_x = U'(x) integrate to H = p^2/2m + U
H_sym = p^2/(2*m) + m*omega^2/2*q^2   # formal: p is the Xi symbol, q the P symbol
assert (diff(H_sym, p) - p/m).simplify_full().is_zero(),     'H_p = p/m FAILED'
assert (diff(H_sym, q) - m*omega^2*q).simplify_full().is_zero(), "H_x = U' FAILED"
print("OK: H_p = p/m and H_x = U'(x) integrate to H = p^2/2m + U")

# Classical case: a generator L(x_hat, p_hat) commutes with x_hat
L_classical_obstruction = lambda f: (x^2 + p)*f   # a function of (x, p) only
lhs_obs = comm_op(L_classical_obstruction, x_hat_Xi, psi_Xi)
assert lhs_obs.is_zero(), f'obstruction FAILED: got {lhs_obs}'
print("OK: [L(x,p), x_hat] = 0, so no function of x, p alone gives im[L, x_hat] = p")

OK: [H_osc, q_hat] = -hbar^2/m * d/dq  (= -i*hbar*p_hat/m)
OK: [H_osc, p_hat] = i*hbar*U'(q)
OK: H_p = p/m and H_x = U'(x) integrate to H = p^2/2m + U
OK: [L(x,p), x_hat] = 0, so no function of x, p alone gives im[L, x_hat] = p


## 5. The Liouvillian and the Liouville equation

With the classical algebra of section 4, the chain rule gives $L'_{\lambda_x}=p/m$ and $L'_{\lambda_p}=-U'(x)$, so $\hat L=(\hat p/m)\hat\lambda_x-U'(\hat x)\hat\lambda_p$ up to an additive function of $(x,p)$, which is set to zero. In the $(x,p)$ representation, $i\partial_t\Psi=\hat L\Psi$ gives the Liouville equation $\partial_t\rho=-\{H,\rho\}_{\mathcal P}$.

In [6]:
psi_Xi_xp = function('psi_Xi_xp')(x, p)   # fresh test function on Xi

# L_osc on Xi: (p/m)*lx_hat - m*omega^2*x*lp_hat
L_osc_Xi = lambda f: (p/m)*lx_hat(f) + (-m*omega^2*x)*lp_hat(f)

# (a) [L, x_hat] = -i*(p/m) = -i*L'_{lambda_x}
lhs_La = comm_op(L_osc_Xi, x_hat_Xi, psi_Xi_xp)
rhs_La = (-I*p/m * psi_Xi_xp).simplify_full()
assert (lhs_La - rhs_La).simplify_full().is_zero(), \
    f'[L, x_hat] FAILED: {lhs_La} != {rhs_La}'
print("OK: [L_osc, x_hat] = -i*(p/m)  (= -i*L'_{lambda_x})")

# (b) [L, p_hat] = i*U'(x) = -i*L'_{lambda_p}
lhs_Lb = comm_op(L_osc_Xi, p_hat_Xi, psi_Xi_xp)
rhs_Lb = (I*m*omega^2*x * psi_Xi_xp).simplify_full()   # i*U'(x) = i*m*omega^2*x
assert (lhs_Lb - rhs_Lb).simplify_full().is_zero(), \
    f'[L, p_hat] FAILED: {lhs_Lb} != {rhs_Lb}'
print("OK: [L_osc, p_hat] = i*U'(x)  (= -i*L'_{lambda_p})")

# (c) L'_{lambda_x} = p/m and L'_{lambda_p} = -U'(x) integrate to L = (p/m)*lambda_x - U'*lambda_p
L_sym = (p/m)*lambda_x - m*omega^2*x*lambda_p
assert (diff(L_sym, lambda_x) - p/m).simplify_full().is_zero(),  'L_{lambda_x} FAILED'
assert (diff(L_sym, lambda_p) + m*omega^2*x).simplify_full().is_zero(), 'L_{lambda_p} FAILED'
print("OK: L'_{lambda_x} = p/m and L'_{lambda_p} = -U' integrate to L = (p/m)*lambda_x - U'*lambda_p")

# (d) Liouville equation. Poisson bracket on P, written in the symbols (x, p):
def poisson_P(f, g):
    return diff(f, p)*diff(g, x) - diff(f, x)*diff(g, p)

H_phys = p^2/(2*m) + m*omega^2/2*x^2
rho    = function('rho')(x, p)

# {H,rho}_P = (p/m)*d_x rho - m*omega^2*x*d_p rho
brack_HP = poisson_P(H_phys, rho).simplify_full()
exp_brack = (p/m*diff(rho,x) - m*omega^2*x*diff(rho,p)).simplify_full()
assert (brack_HP - exp_brack).simplify_full().is_zero(), \
    f'Liouville bracket FAILED: {brack_HP}'

# i*d_t psi = L_hat*psi  =>  d_t rho = -i*L_hat rho
#   = -i*[-i*(p/m)*d_x rho + i*m*omega^2*x*d_p rho]
#   = -(p/m)*d_x rho + m*omega^2*x*d_p rho  = -{H,rho}_P
L_action = (-I * L_osc_Xi(rho)).simplify_full()
liouv_rhs = (-brack_HP).simplify_full()
assert (L_action - liouv_rhs).simplify_full().is_zero(), \
    f'Liouville eq FAILED: -i*L_hat rho={L_action}, expected {liouv_rhs}'
print("OK: d_t rho = -i*L_hat rho = -{H,rho}_P = -(p/m)*d_x rho + U'*d_p rho")

OK: [L_osc, x_hat] = -i*(p/m)  (= -i*L'_{lambda_x})
OK: [L_osc, p_hat] = i*U'(x)  (= -i*L'_{lambda_p})
OK: L'_{lambda_x} = p/m and L'_{lambda_p} = -U' integrate to L = (p/m)*lambda_x - U'*lambda_p
OK: d_t rho = -i*L_hat rho = -{H,rho}_P = -(p/m)*d_x rho + U'*d_p rho


## Final check

The main identities once more, in one cell.

In [7]:
# Chain rule
assert (comm_op(lambda f: q^2*f, p_hat_P, psi_P)
        - I*hbar*2*q*psi_P).simplify_full().is_zero(), 'A1'
assert (comm_op(p_hat_P, q_hat_P, psi_P)
        - (-I*hbar*psi_P)).simplify_full().is_zero(),    'A2'
assert (comm_op(lambda f: x^2*f, lx_hat, psi_Xi)
        - I*2*x*psi_Xi).simplify_full().is_zero(),       'A3'

# Quantum Hamiltonian and classical obstruction
assert (comm_op(H_osc_P, p_hat_P, psi_P)
        - I*hbar*m*omega^2*q*psi_P).simplify_full().is_zero(), '[H, p_hat]'
assert comm_op(L_classical_obstruction, x_hat_Xi, psi_Xi).is_zero(), \
    'classical obstruction'

# Liouvillian
assert (comm_op(L_osc_Xi, x_hat_Xi, psi_Xi_xp)
        - (-I*p/m*psi_Xi_xp)).simplify_full().is_zero(), '[L, x_hat]'
assert (comm_op(L_osc_Xi, p_hat_Xi, psi_Xi_xp)
        - I*m*omega^2*x*psi_Xi_xp).simplify_full().is_zero(), '[L, p_hat]'

# Bracket of the kappa-identification
assert (poisson_Xi(x_q, p_q).simplify_full() - hbar*kappa).is_zero(), \
    '{x_q,p_q} = hbar*kappa'

print('C1: all checks passed.')

C1: all checks passed.
